In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')



In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Task 1: Write your code here:
df = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')

print(f"Dataset shape: {df.shape}")
df.head()


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
missing_values = df.isnull()
missing_cols = missing_values.columns
missing_cols

In [ ]:
# Task 2: Write your code here:
# Select relevant columns, we do not select 'model' (too many unique values, too sparse and will hurt the model performance)
cols = missing_cols
df_clean = df[cols].copy()

# Drop rows where target (price) or key features are missing - can't predict without them
print(f"Before: {df_clean.shape}")
df_clean = df[cols].fillna(df[cols].mean())
print(f"After dropping missing price/year/odometer: {df_clean.shape}")

In [ ]:
# Task 1: Write your code here:

def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(df_clean)

In [ ]:
# Task 2: Write your code here:

# 4. Do we have duplicate samples?
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:
df_clean.info()

In [ ]:
# Task 4: Write your code here:

feature_cols = df_clean.drop("Target", axis=1).astype(float)



# Standardize features using StandardScaler
numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  ### DON'T SCALE THE TARGET
scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean


In [ ]:
import seaborn as sns


In [ ]:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Target")

In [ ]:
#it is imbalanced so we use StratifiedKFold

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop("Target", axis=1).astype(float)
y = df_clean['Target'].astype(float)
X


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostClassifier


In [ ]:
# Task 2,3,4,5: Write your code here:


n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for logistic regression results for each fold
lr_losses = []
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

In [ ]:
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )


for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model.fit(X_train, y_train)
        # Predict on the test set
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
f1

In [ ]:
print(f'the accuracy{accuracy}')

In [ ]:
# Task 1: Write your code here:

# Retrieve CatBoost feature importances and sort them
catboost_model = model
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()


In [ ]:
catboost_importance = list(zip(X.columns, catboost_model.feature_importances_))
catboost_importance[0]


In [ ]:
#Print the averaged score across all folds

In [ ]:
total = 0
for i in lr_f1:
  total += i


print('the average f1 across folds: ', total / 5)

In [ ]:
total = 0
for i in lr_accuracy:
  print(i)
  total += i
print('the average accuracy across folds: ', total / 5)

In [ ]:
# Task Bonus: Write your code here:
newx = df_clean['P_2']
newx

In [ ]:
model = CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
lr_accuracy = []
lr_precision = []
lr_recall = []
lr_f1 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(newx, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
    model.fit(X_train, y_train)
        # Predict on the test set
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    lr_accuracy.append(accuracy)
    lr_precision.append(precision)
    lr_recall.append(recall)
    lr_f1.append(f1)

In [ ]:
total = 0
for i in lr_f1:
  total += i


print('the average f1 across folds: ', total / 5)

In [ ]:
total = 0
for i in lr_accuracy:
  print(i)
  total += i
print('the average accuracy across folds: ', total / 5)